# Housing Data Cleaning + Cleaning Log
Run the cells **top to bottom**. Every row in the cleaning log is created by the code
as it inspects the data (nothing is typed in by hand), and the log is written to
`cleaning-log.csv` at the end. Cleaned data goes to `cleaned.csv`.

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd

SRC = "../data/raw/train.csv"
OUTDIR = "../data/processed/"

print('Finished Setup')

Finished Setup


## 2. Rules and schema

In [2]:
# Numeric columns (continuous, counts, years, ordinal scores)
NUMERIC = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "MoSold", "YrSold", "SalePrice",
]

# In these columns the text "NA" means "feature not present", not "value missing"
STRUCTURAL_NA = {
    "Alley": "NoAlley",
    "BsmtQual": "NoBasement", "BsmtCond": "NoBasement",
    "BsmtExposure": "NoBasement", "BsmtFinType1": "NoBasement",
    "BsmtFinType2": "NoBasement",
    "FireplaceQu": "NoFireplace",
    "GarageType": "NoGarage", "GarageFinish": "NoGarage",
    "GarageQual": "NoGarage", "GarageCond": "NoGarage",
    "PoolQC": "NoPool",
    "Fence": "NoFence",
    "MiscFeature": "None",
}

# Known label inconsistencies vs. the data dictionary
LABEL_FIXES = {
    "Exterior1st": {"Wd Shng": "WdShing", "CmentBd": "CemntBd", "Brk Cmn": "BrkComm"},
    "Exterior2nd": {"Wd Shng": "WdShing", "CmentBd": "CemntBd", "Brk Cmn": "BrkComm"},
    "BldgType": {"2fmCon": "2FmCon", "Duplex": "Duplx", "Twnhs": "TwnhsI"},
}

# Columns checked for statistical outliers (IQR rule), and the minimum sample size
OUTLIER_COLS = ["LotFrontage", "LotArea", "TotalBsmtSF", "1stFlrSF",
                "GrLivArea", "GarageArea", "SalePrice"]
MIN_N_OUTLIER = 30

# Hard validity ranges (violations are excluded)
RANGES = {"OverallQual": (1, 10), "OverallCond": (1, 10), "MoSold": (1, 12)}

print('Finished Rules and Schema')

Finished Rules and Schema


## 3. Load data and set up the log

In [3]:
raw = pd.read_csv(SRC, dtype=str, keep_default_na=False)
raw.columns = raw.columns.str.strip()
df = raw.copy()
n_processed = len(df)

LOG_COLS = ["Record ID", "Field", "Original Value",
            "Cleaning Decision", "Reason"]
log = []   # filled by the cleaning steps below
excluded = set()   # row indices removed from the cleaned data

def add(rid, field, orig, decision, reason):
    """Record one cleaning decision."""
    log.append([rid, field, orig, decision, reason])

print(f"Loaded {n_processed} records, {df.shape[1]} columns")
df.head()

print('Finished Load Data and Setup Log')

Loaded 1460 records, 81 columns
Finished Load Data and Setup Log


## 4. Trim whitespace

In [4]:
for col in df.columns:
    trimmed = df[col].str.strip()
    for i in df.index[trimmed != df[col]]:
        add(df.at[i, "Id"], col, repr(df.at[i, col]), "Standardized",
            "Removed leading/trailing whitespace")
    df[col] = trimmed

print('Finished Whitespace Trim')

Finished Whitespace Trim


## 5. Duplicate records

In [5]:
dup_rows = df.duplicated(keep="first")
n_dups = int(dup_rows.sum())
for i in df.index[dup_rows]:
    add(df.at[i, "Id"], "(entire row)", "duplicate", "Excluded",
        "Exact duplicate of an earlier record")
df = df[~dup_rows]

for i in df.index[df["Id"].duplicated(keep=False)]:
    add(df.at[i, "Id"], "Id", df.at[i, "Id"], "Flagged",
        "Duplicate Id with different content; needs manual review")

print('Finished Cleaning Duplicated Records')

Finished Cleaning Duplicated Records


## 6. Standardize inconsistent labels

In [6]:
for col, mapping in LABEL_FIXES.items():
    if col not in df:
        continue
    for bad, good in mapping.items():
        for i in df.index[df[col] == bad]:
            add(df.at[i, "Id"], col, f'"{bad}"', "Standardized",
                f'Standardized to "{good}" to match the data dictionary '
                f"and the other exterior/building columns")
            df.at[i, col] = good

print('Finished Standardizing Inconisistent Labels')

Finished Standardizing Inconisistent Labels


## 7. Structural "NA" → explicit label

In [7]:
for col, label in STRUCTURAL_NA.items():
    if col not in df:
        continue
    for i in df.index[df[col] == "NA"]:
        add(df.at[i, "Id"], col, '"NA"', "Recoded",
            f'"NA" means feature absent in the data dictionary; '
            f'recoded to "{label}" so it is not read as missing')
        df.at[i, col] = label

print('Finished Structural "NA" → explicit label')

Finished Structural "NA" → explicit label


## 8. Convert numeric columns

In [8]:
for col in NUMERIC:
    if col not in df:
        continue
    num = pd.to_numeric(df[col].replace("NA", np.nan), errors="coerce")
    bad = df[col].ne("NA") & df[col].ne("") & num.isna()
    for i in df.index[bad]:
        add(df.at[i, "Id"], col, f'"{df.at[i, col]}"', "Set to missing",
            "Non-numeric text in a numeric field")
    df[col] = num

print('Finished Convert numeric columns')

Finished Convert numeric columns


## 9. Handle missing numeric values

In [9]:
# LotFrontage: median of the same neighborhood, else overall median
overall_med = df["LotFrontage"].median()
nb_med = df.groupby("Neighborhood")["LotFrontage"].transform("median")
for i in df.index[df["LotFrontage"].isna()]:
    val, basis = nb_med[i], "neighborhood median"
    if pd.isna(val):
        val, basis = overall_med, "overall median (no other homes in neighborhood)"
    val = int(round(val))
    add(df.at[i, "Id"], "LotFrontage", "NA", "Imputed",
        f"Missing; imputed {val} ft using {basis}")
    df.at[i, "LotFrontage"] = val

# MasVnrArea: missing + no veneer -> 0
for i in df.index[df["MasVnrArea"].isna()]:
    if df.at[i, "MasVnrType"] in ("None", "NA", ""):
        add(df.at[i, "Id"], "MasVnrArea", "NA", "Imputed",
            "No masonry veneer, so area set to 0")
        df.at[i, "MasVnrArea"] = 0

# Basement / garage numeric fields where the feature is absent -> 0
for col in ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
            "BsmtFullBath", "BsmtHalfBath"]:
    for i in df.index[df[col].isna() & (df["BsmtQual"] == "NoBasement")]:
        add(df.at[i, "Id"], col, "NA", "Imputed", "No basement, so set to 0")
        df.at[i, col] = 0
for col in ["GarageCars", "GarageArea"]:
    for i in df.index[df[col].isna() & (df["GarageType"] == "NoGarage")]:
        add(df.at[i, "Id"], col, "NA", "Imputed", "No garage, so set to 0")
        df.at[i, col] = 0

# GarageYrBlt stays blank when there is no garage (a year of 0 would mislead)
for i in df.index[df["GarageYrBlt"].isna() & (df["GarageType"] == "NoGarage")]:
    add(df.at[i, "Id"], "GarageYrBlt", "NA", "Confirmed",
        "No garage; left blank (imputing a year would be misleading)")

print('Finished Handle missing numeric values')

Finished Handle missing numeric values


## 10. Integrity checks

In [10]:
for col, (lo, hi) in RANGES.items():
    for i in df.index[(df[col] < lo) | (df[col] > hi)]:
        add(df.at[i, "Id"], col, df.at[i, col], "Excluded",
            f"Outside valid range {lo}-{hi}")
        excluded.add(i)

for i in df.index[df["SalePrice"].isna() | (df["SalePrice"] <= 0)]:
    add(df.at[i, "Id"], "SalePrice", df.at[i, "SalePrice"], "Excluded",
        "Missing or non-positive target value")
    excluded.add(i)

for i in df.index[df["YearBuilt"] > df["YrSold"]]:
    add(df.at[i, "Id"], "YearBuilt", df.at[i, "YearBuilt"], "Flagged",
        "Built after sale year; review (could be a pre-construction sale)")

for i in df.index[df["YearRemodAdd"] < df["YearBuilt"]]:
    add(df.at[i, "Id"], "YearRemodAdd", df.at[i, "YearRemodAdd"], "Corrected",
        f"Remodel year before build year; set to YearBuilt ({df.at[i, 'YearBuilt']})")
    df.at[i, "YearRemodAdd"] = df.at[i, "YearBuilt"]

above_grade = df["1stFlrSF"] + df["2ndFlrSF"] + df["LowQualFinSF"]
for i in df.index[df["GrLivArea"] != above_grade]:
    add(df.at[i, "Id"], "GrLivArea", df.at[i, "GrLivArea"], "Flagged",
        "GrLivArea != 1stFlrSF + 2ndFlrSF + LowQualFinSF")

bsmt_sum = df["BsmtFinSF1"] + df["BsmtFinSF2"] + df["BsmtUnfSF"]
for i in df.index[df["TotalBsmtSF"] != bsmt_sum]:
    add(df.at[i, "Id"], "TotalBsmtSF", df.at[i, "TotalBsmtSF"], "Flagged",
        "TotalBsmtSF != BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF")

for i in df.index[df["TotRmsAbvGrd"] < df["BedroomAbvGr"] + df["KitchenAbvGr"]]:
    add(df.at[i, "Id"], "TotRmsAbvGrd", df.at[i, "TotRmsAbvGrd"], "Flagged",
        "Fewer total rooms than bedrooms + kitchens")

# Rules that passed for every record -> one summary confirmation each
def passed(field, reason):
    if not any(r[2] == field and r[4] in ("Flagged", "Excluded", "Corrected") for r in log):
        add("ALL", field, "-", "Confirmed", reason)

passed("GrLivArea", "GrLivArea = 1stFlrSF + 2ndFlrSF + LowQualFinSF for every record")
passed("TotalBsmtSF", "TotalBsmtSF = BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF for every record")
passed("YearRemodAdd", "Remodel year >= build year for every record")
passed("MoSold", "Sale month within 1-12 for every record")
passed("OverallQual", "Quality/condition ratings within 1-10 for every record")
passed("SalePrice", "All sale prices present and positive")

print('Finished Integrity checks')

Finished Integrity checks


## 11. Outliers (IQR rule): keep, but document

In [11]:
for col in OUTLIER_COLS:
    s = df[col].dropna()
    if len(s) < MIN_N_OUTLIER:
        add("ALL", col, "-", "Confirmed",
            f"Outlier test skipped: only {len(s)} records (IQR needs >= {MIN_N_OUTLIER}); "
            f"no values are physically impossible")
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    for i in df.index[(df[col] < lo) | (df[col] > hi)]:
        add(df.at[i, "Id"], col, f"{df.at[i, col]:,.0f}", "Kept (Outlier)",
            f"Outside IQR fence [{lo:,.0f}, {hi:,.0f}]; value is plausible "
            f"(no sign of entry error), so retained")

print('Finished Outliers')

Finished Outliers


## 12. Finalize: drop excluded rows, fix types

In [12]:
df = df.drop(index=list(excluded))

for col in NUMERIC:
    if (df[col].dropna() % 1 == 0).all():
        df[col] = df[col].astype("Int64")

df["MSSubClass"] = df["MSSubClass"].astype(str)   # nominal code, not a quantity
df["Id"] = df["Id"].astype(int)
df = df.sort_values("Id")

print('Finished Finalize')

Finished Finalize


## 13. Build the cleaning log from the recorded decisions and save both files

In [13]:
log_df = pd.DataFrame(log, columns=LOG_COLS)

# Sort by record number; "ALL" (dataset-wide) rows go last
log_df["_s"] = pd.to_numeric(log_df["Record ID"], errors="coerce").fillna(1e9)
log_df = log_df.sort_values("_s", kind="stable").drop(columns="_s")

# "Flagged" = records that needed a real data-quality decision
# (routine NA recoding and confirmations don't count)
issue = {"Standardized", "Imputed", "Corrected", "Kept (Outlier)",
         "Flagged", "Excluded", "Set to missing"}
is_issue = log_df["Cleaning Decision"].isin(issue) & (log_df["Record ID"] != "ALL")
n_flagged = log_df.loc[is_issue, "Record ID"].nunique()
n_excluded = len(excluded) + n_dups

summary = (f"Total records processed: {n_processed:,}. "
           f"Total flagged records: {n_flagged}. "
           f"Total records excluded: {n_excluded}.")
log_df.loc[len(log_df)] = ["ALL", "SUMMARY", "-", "Summary", summary]

os.makedirs(OUTDIR, exist_ok=True)
df.to_csv(os.path.join(OUTDIR, "cleaned.csv"), index=False)
log_df.to_csv(os.path.join(OUTDIR, "cleaning-log.csv"), index=False)

print(summary)
print(f"cleaned.csv: {len(df)} rows | cleaning-log.csv: {len(log_df)} rows")
print('Finished Build the cleaning log')

Total records processed: 1,460. Total flagged records: 593. Total records excluded: 0.
cleaned.csv: 1460 rows | cleaning-log.csv: 7550 rows
Finished Build the cleaning log


## 14. Preview

In [14]:
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
log_df[log_df["Cleaning Decision"] != "Recoded"]   # hides the routine NA recodes; remove filter to see all

,Record ID,Field,Original Value,Cleaning Decision,Reason
0,4,Exterior2nd,"""Wd Shng""",Standardized,"Standardized to ""WdShing"" to match the data dictionary and the other exterior/building columns"
6839,8,LotFrontage,NA,Imputed,Missing; imputed 80 ft using neighborhood median
1,9,Exterior2nd,"""Wd Shng""",Standardized,"Standardized to ""WdShing"" to match the data dictionary and the other exterior/building columns"
105,10,BldgType,"""2fmCon""",Standardized,"Standardized to ""2FmCon"" to match the data dictionary and the other exterior/building columns"
2,12,Exterior2nd,"""Wd Shng""",Standardized,"Standardized to ""WdShing"" to match the data dictionary and the other exterior/building columns"
...,...,...,...,...,...
7189,ALL,YearRemodAdd,-,Confirmed,Remodel year >= build year for every record
7190,ALL,MoSold,-,Confirmed,Sale month within 1-12 for every record
7191,ALL,OverallQual,-,Confirmed,Quality/condition ratings within 1-10 for every record
7192,ALL,SalePrice,-,Confirmed,All sale prices present and positive


In [15]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65,8450,Pave,NoAlley,Reg,Lvl,AllPub,...,0,NoPool,NoFence,None,0,2,2008,WD,Normal,208500
1,2,20,RL,80,9600,Pave,NoAlley,Reg,Lvl,AllPub,...,0,NoPool,NoFence,None,0,5,2007,WD,Normal,181500
2,3,60,RL,68,11250,Pave,NoAlley,IR1,Lvl,AllPub,...,0,NoPool,NoFence,None,0,9,2008,WD,Normal,223500
3,4,70,RL,60,9550,Pave,NoAlley,IR1,Lvl,AllPub,...,0,NoPool,NoFence,None,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84,14260,Pave,NoAlley,IR1,Lvl,AllPub,...,0,NoPool,NoFence,None,0,12,2008,WD,Normal,250000
